In [2]:
%cd /run/media/fourier/Data1/Pras/Thesis_Nexus/NXSCough

import argparse, inspect, json, os, pickle, socket, subprocess, warnings, random, math, librosa, shutil
import numpy as np
import pandas as pd
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import lightning as L
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from lightning.pytorch.callbacks import ModelCheckpoint
from lightning.pytorch.loggers.tensorboard import TensorBoardLogger
from lightning.pytorch.callbacks.early_stopping import EarlyStopping

import opensmile
import pandas as pd
import numpy as np
import soundfile as sf
from sklearn.preprocessing import StandardScaler
from pathlib import Path
import joblib

import opensmile
import pandas as pd
from pathlib import Path

from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import confusion_matrix, accuracy_score, balanced_accuracy_score
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib import cm

import train
import commons
import models
import utils
import lightning_wrapper
from cough_datasets import CoughDatasets, CoughDatasetsCollate
from IPython.display import Audio

torch.set_float32_matmul_precision("medium")
cmap = cm.get_cmap("viridis")

/run/media/fourier/Data1/Pras/Thesis_Nexus/NXSCough


/tmp/ipykernel_2994891/233994203.py:46: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  cmap = cm.get_cmap("viridis")


In [3]:
parser = train.parse_args()
args = parser.parse_args(["--init", "--model_name", "dev", "--pooling_model",
                          "opensmiledev", "--config_path", "configs/general.json"]) # qwen

model_dir = os.path.join("./logs", args.model_name)
os.makedirs(model_dir, exist_ok=True)
port = None

config_path = args.config_path if args.init else os.path.join(model_dir, "config.json")
hps = train.load_config(config_path, model_dir, args)

df_train, _ = train.load_data(hps)
#df_train = df_train[df_train['disease_status'] == 1].reset_index(drop=True)
collate_fn = train.get_collate_fn(hps)
target_labels = df_train[hps.data.target_column]

In [4]:
example_data = df_train.sample(n=1)[['path_file']].values[0][0]

In [5]:
FEATURE_SETS = [
    opensmile.FeatureSet.ComParE_2016,
    opensmile.FeatureSet.GeMAPSv01b,
    opensmile.FeatureSet.eGeMAPSv02,
    opensmile.FeatureSet.emobase,
    # opensmile.FeatureSet.IS09,
    # opensmile.FeatureSet.IS10,
    # opensmile.FeatureSet.IS11,
    # opensmile.FeatureSet.IS12,
    # opensmile.FeatureSet.IS13,
]

SMILE_CLIENTS = {
    str(fs): opensmile.Smile(
        feature_set=fs,
        feature_level=opensmile.FeatureLevel.Functionals # LowLevelDescriptors Functionals
    )
    for fs in FEATURE_SETS
}

def peak_normalize(x, target_peak=0.9, eps=1e-8):
    x = np.asarray(x, dtype=np.float32)

    peak = np.max(np.abs(x))
    if peak < 1e-6:
        return x

    x = x * (target_peak / (peak + eps))
    return x

def load_and_normalize(wav_path):
    audio, sr = sf.read(wav_path)
    audio = peak_normalize(audio)
    return audio, sr

def extract_all_features(wav_path, audio, sr):
    outputs = []

    for fs_name, client in SMILE_CLIENTS.items():
        df = client.process_signal(audio, sr)
        df = df.reset_index(drop=True)
        df = df.add_prefix(fs_name + "__")
        outputs.append(df)

    merged = pd.concat(outputs, axis=1)
    merged = merged.fillna(0.0)
    merged["path_file"] = wav_path

    return merged

In [6]:
from concurrent.futures import ProcessPoolExecutor, as_completed
from tqdm import tqdm
import pandas as pd

def process_file(filepath):
    audio, sr = load_and_normalize(filepath)
    f = extract_all_features(filepath, audio, sr)
    return f.fillna(0.0)

def parallel_extract(filepaths, n_workers=None):
    results = []

    with ProcessPoolExecutor(max_workers=n_workers) as executor:
        futures = [executor.submit(process_file, fp) for fp in filepaths]

        for f in tqdm(as_completed(futures), total=len(futures)):
            results.append(f.result())

    return pd.concat(results, ignore_index=True)

all_dfs = parallel_extract(df_train['path_file'].tolist(), n_workers=8)
all_dfs = all_dfs.merge(
    df_train[["path_file", "disease_status", "participant"]],
    on="path_file",
    how="left"
)

100%|██████████| 9279/9279 [01:49<00:00, 84.89it/s] 


In [7]:
X = all_dfs.drop(columns=["path_file", "disease_status", "participant"]).values
y = all_dfs["disease_status"].values
groups = all_dfs["participant"].values

In [9]:
scaler = StandardScaler()
scaler.fit(X)

,copy,True
,with_mean,True
,with_std,True


In [ ]:
# from joblib import dump
# dump(scaler, "/run/media/fourier/Data1/Pras/Thesis_Nexus/NXSCough/precomputed_features/opensmile/scaler.joblib")

['/run/media/fourier/Data1/Pras/Thesis_Nexus/NXSCough/precomputed_features/opensmile/scaler.joblib']

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold
from xgboost import XGBClassifier
import numpy as np

sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)

feature_importances = []
for train_idx, val_idx in tqdm(sgkf.split(X, y, groups), total=5):
    X_train, y_train = X[train_idx], y[train_idx]

    model = XGBClassifier(
        n_estimators=300,
        max_depth=4,
        colsample_bytree=0.3,
        subsample=0.8,
        random_state=42,
        n_jobs=-1
    )

    model.fit(X_train, y_train)
    feature_importances.append(model.feature_importances_)

importances = np.mean(feature_importances, axis=0)

threshold = np.percentile(importances, 75)  # top 25%
selected_idx = np.where(importances >= threshold)[0]
X_selected = X[:, selected_idx]

100%|██████████| 5/5 [02:04<00:00, 25.00s/it]


In [34]:
threshold = np.percentile(importances, 95)  # top 25%
selected_idx = np.where(importances >= threshold)[0]
X_selected = X[:, selected_idx]

In [35]:
X_selected.shape

(9279, 327)

In [37]:
all_dfs.columns[selected_idx]

Index(['FeatureSet.ComParE_2016__audspec_lengthL1norm_sma_range',
       'FeatureSet.ComParE_2016__audspec_lengthL1norm_sma_percentile99.0',
       'FeatureSet.ComParE_2016__pcm_RMSenergy_sma_iqr1-3',
       'FeatureSet.ComParE_2016__pcm_RMSenergy_sma_stddev',
       'FeatureSet.ComParE_2016__audspec_lengthL1norm_sma_de_range',
       'FeatureSet.ComParE_2016__audspec_lengthL1norm_sma_de_percentile1.0',
       'FeatureSet.ComParE_2016__audspec_lengthL1norm_sma_de_pctlrange0-1',
       'FeatureSet.ComParE_2016__audspec_lengthL1norm_sma_de_kurtosis',
       'FeatureSet.ComParE_2016__audspecRasta_lengthL1norm_sma_de_range',
       'FeatureSet.ComParE_2016__audspecRasta_lengthL1norm_sma_de_iqr1-3',
       ...
       'FeatureSet.ComParE_2016__mfcc_sma_de[10]_peakMeanRel',
       'FeatureSet.GeMAPSv01b__F0semitoneFrom27.5Hz_sma3nz_amean',
       'FeatureSet.GeMAPSv01b__hammarbergIndexUV_sma3nz_amean',
       'FeatureSet.eGeMAPSv02__F0semitoneFrom27.5Hz_sma3nz_amean',
       'FeatureSet.eGeMA

In [36]:
from sklearn.ensemble import RandomForestClassifier

clf = RandomForestClassifier(n_estimators=100)
clf.fit(X_selected, groups)

print(clf.score(X_selected, groups))

1.0


In [23]:
from sklearn.feature_selection import VarianceThreshold

vt = VarianceThreshold(threshold=1e-5)
X = vt.fit_transform(stacked)

In [24]:
X.shape

(9279, 6464)

In [ ]:
import joblib
joblib.dump(scaler, "precomputed_stats/opensmile_global_scaler.pkl")
scaler = joblib.load("precomputed_stats/opensmile_global_scaler.pkl")

In [ ]:
f = extract_all_features(wav)
f = f.fillna(0.0)
f = f.drop(columns=["filepath"]).values

In [ ]:
f.shape

In [ ]:
scaler.transform(f)


In [ ]:
stacked.values.shape

In [ ]:
def fit_global_scaler(all_feature_frames):
    stacked = pd.concat(
        [df.drop(columns=["filepath"]) for df in all_feature_frames],
        axis=0,
        ignore_index=True
    )

    scaler = StandardScaler()
    scaler.fit(stacked.values)
    return scaler

In [ ]:
feats = extract_all_features(str(example_data))

In [ ]:
feats.shape

In [ ]:
feats.columns